# AlgoPay SDK — Controllability & Observability Demo

This notebook demonstrates **every core feature** of the AlgoPay SDK:
- Wallet creation on Algorand TestNet
- 6 guard types (budget, single-tx, rate limit, justification, recipient whitelist, confirm)
- Payment execution with guard enforcement
- Guard blocks with detailed reasons
- Payment intents (human-in-the-loop authorize/capture)
- Batch payments
- Ledger audit trail
- **Telemetry → Dashboard**: every event streams to the hosted console

**After running this notebook, switch to the dashboard** at your `ALGOPAY_CONSOLE_URL` to see the SDK Events page light up.

---

**Requirements:** `pip install algopay-sdk pandas matplotlib`

In [ ]:
# Cell 0 — Setup & Configuration
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "algopay-sdk", "pandas", "matplotlib"])

import os, getpass

# Configure dashboard telemetry
if not os.environ.get("ALGOPAY_CONSOLE_URL"):
    os.environ["ALGOPAY_CONSOLE_URL"] = input("Dashboard URL (e.g. https://algopay-sdk-pay-b17a.vercel.app): ").strip()
if not os.environ.get("ALGOPAY_API_KEY"):
    os.environ["ALGOPAY_API_KEY"] = getpass.getpass("API Key (sk_live_...): ")

from algopay import AlgoPay
from algopay.core.types import Network, PaymentRequest, FeeLevel
import pandas as pd
from decimal import Decimal

RECIPIENT = "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAY5HFKQ"

client = AlgoPay(network=Network.ALGORAND_TESTNET)
print(f"AlgoPay initialized (network={client.config.network.value})")
print(f"Telemetry enabled: {client.telemetry.enabled}")
print(f"Console URL: {os.environ.get('ALGOPAY_CONSOLE_URL', 'not set')}")

In [ ]:
# Cell 1 — Create Wallet on TestNet
wallet_set = await client.create_wallet_set("demo-semifinal")
wallet = await client.create_wallet(wallet_set_id=wallet_set.id)

print(f"Wallet Set: {wallet_set.id} ({wallet_set.name})")
print(f"Wallet ID:  {wallet.id}")
print(f"Address:    {wallet.address}")
print(f"Network:    {wallet.blockchain}")

try:
    balance = await client.get_balance(wallet.id)
    print(f"USDC Balance: {balance}")
except Exception:
    print("USDC Balance: 0.00 (wallet not funded yet — guards still work)")

In [ ]:
# Cell 2 — Configure Guards (Controllability)
# These guards protect the wallet before ANY payment can execute.

await client.add_budget_guard(wallet.id, daily_limit="0.10", name="daily_budget")
await client.add_single_tx_guard(wallet.id, max_amount="0.05", name="single_tx_cap")
await client.add_rate_limit_guard(wallet.id, max_per_hour=5, name="hourly_rate")
await client.add_justification_guard(wallet.id, min_length=3, name="justification")
await client.add_recipient_guard(
    wallet.id,
    mode="whitelist",
    addresses=[RECIPIENT],
    name="recipient_allowlist",
)

guards = await client.list_guards(wallet.id)
print("Active guards:")
for g in guards:
    print(f"  ✓ {g}")
print(f"\nTotal: {len(guards)} guards protecting wallet {wallet.id[:8]}...")

In [ ]:
# Cell 3 — Happy Path Payment
# This payment passes ALL guards: within budget, under single-tx cap,
# has justification, and recipient is whitelisted.

result = await client.pay(
    wallet_id=wallet.id,
    recipient=RECIPIENT,
    amount="0.01",
    purpose="Weather API lookup for Tokyo forecast",
)

print(f"Success:    {result.success}")
print(f"Status:     {result.status.value}")
print(f"Amount:     {result.amount} USDC")
print(f"TX Hash:    {result.blockchain_tx}")
print(f"Guards OK:  {result.guards_passed}")
print(f"\n→ Dashboard received: payment.initiated → payment.guard_passed → payment.completed")

In [ ]:
# Cell 4 — Guard Block: Over Budget
# The daily budget is $0.10 — this $0.50 payment will be BLOCKED.

over_budget = await client.pay(
    wallet_id=wallet.id,
    recipient=RECIPIENT,
    amount="0.50",
    purpose="Expensive market data feed",
)

print(f"Success:  {over_budget.success}")
print(f"Status:   {over_budget.status.value}")
print(f"Error:    {over_budget.error}")
print(f"\n→ Dashboard received: payment.initiated → payment.guard_blocked")
print(f"  Block reason visible on SDK Events page")

In [ ]:
# Cell 5 — Guard Block: Missing Justification
# JustificationGuard requires a non-empty purpose string.

no_purpose = await client.pay(
    wallet_id=wallet.id,
    recipient=RECIPIENT,
    amount="0.01",
    purpose="",  # Empty — blocked!
)

print(f"Success:  {no_purpose.success}")
print(f"Status:   {no_purpose.status.value}")
print(f"Error:    {no_purpose.error}")

In [ ]:
# Cell 6 — Guard Block: Recipient Not Whitelisted
# Only RECIPIENT is in the allowlist. This unknown address is blocked.

UNKNOWN_ADDR = "7ZUILL7ACGQ7G72RH3C2S5UJKB5IHXHI54ZRWLMG76Y347BGOZWBQBKIY"

wrong_recipient = await client.pay(
    wallet_id=wallet.id,
    recipient=UNKNOWN_ADDR,
    amount="0.01",
    purpose="Send to unknown address",
)

print(f"Success:  {wrong_recipient.success}")
print(f"Status:   {wrong_recipient.status.value}")
print(f"Error:    {wrong_recipient.error}")

In [ ]:
# Cell 7 — Guard Block: Rate Limit
# Max 5 per hour. We'll fire rapid payments to trigger the limit.

rate_results = []
for i in range(6):
    r = await client.pay(
        wallet_id=wallet.id,
        recipient=RECIPIENT,
        amount="0.01",
        purpose=f"Rate limit test #{i+1}",
    )
    rate_results.append({"attempt": i + 1, "success": r.success, "status": r.status.value})
    print(f"  Attempt {i+1}: {'✓' if r.success else '✗'} {r.status.value}")

print(f"\nRate limit triggered after {sum(1 for r in rate_results if r['success'])} successful payments")

In [ ]:
# Cell 8 — Payment Intents (Human-in-the-Loop)
# Create → Inspect → Confirm pattern (like Stripe's authorize/capture)

intent = await client.create_payment_intent(
    wallet_id=wallet.id,
    recipient=RECIPIENT,
    amount="0.02",
    purpose="Premium translation service",
)

print(f"Intent ID:     {intent.id}")
print(f"Status:        {intent.status.value}")
print(f"Amount:        {intent.amount} USDC")
print(f"Recipient:     {intent.recipient[:12]}...")
print("\n--- Human reviews and approves ---\n")

# Confirm the intent (capture)
capture = await client.confirm_payment_intent(intent.id)
print(f"Captured:      {capture.success}")
print(f"TX Hash:       {capture.blockchain_tx}")
print(f"Final Status:  {capture.status.value}")

In [ ]:
# Cell 9 — Batch Payments
# Execute 5 payments concurrently. Some will succeed, some will be blocked.

batch_requests = [
    PaymentRequest(wallet_id=wallet.id, recipient=RECIPIENT, amount=Decimal("0.01"), purpose="Batch: search API"),
    PaymentRequest(wallet_id=wallet.id, recipient=RECIPIENT, amount=Decimal("0.01"), purpose="Batch: weather API"),
    PaymentRequest(wallet_id=wallet.id, recipient=RECIPIENT, amount=Decimal("0.01"), purpose="Batch: translate"),
    PaymentRequest(wallet_id=wallet.id, recipient=RECIPIENT, amount=Decimal("0.50"), purpose="Batch: over budget"),
    PaymentRequest(wallet_id=wallet.id, recipient=RECIPIENT, amount=Decimal("0.01"), purpose=""),
]

batch_result = await client.batch_pay(batch_requests, concurrency=3)

print(f"Total:     {batch_result.total}")
print(f"Succeeded: {batch_result.succeeded}")
print(f"Failed:    {batch_result.failed}")
print("\nResults:")
for i, r in enumerate(batch_result.results):
    status = r.status.value if r else "error"
    ok = r.success if r else False
    print(f"  [{i+1}] {'✓' if ok else '✗'} {status}")

In [ ]:
# Cell 10 — Ledger Query & Audit Trail (Observability)
# Every pay() call is recorded in the ledger — success, blocked, or failed.

entries = await client.ledger.query(wallet_id=wallet.id, limit=50)

rows = [
    {
        "status": e.status.value,
        "amount": str(e.amount),
        "purpose": (e.purpose or "")[:40],
        "recipient": (e.recipient or "")[:12] + "...",
        "tx_hash": (e.tx_hash or "")[:12],
        "timestamp": e.timestamp.strftime("%H:%M:%S"),
    }
    for e in entries
]

df = pd.DataFrame(rows)
print(f"Ledger entries: {len(entries)}")
print(f"Completed: {sum(1 for e in entries if e.status.value == 'completed')}")
print(f"Blocked:   {sum(1 for e in entries if e.status.value == 'blocked')}")
print(f"Failed:    {sum(1 for e in entries if e.status.value == 'failed')}")
print()
df

In [ ]:
# Cell 11 — Dashboard Live View
# All the events from this notebook are now visible on the dashboard.

console_url = os.environ.get("ALGOPAY_CONSOLE_URL", "")
events_sent = client.telemetry.events_sent

print("═" * 60)
print("  SWITCH TO DASHBOARD NOW")
print(f"  {console_url}/dashboard/sdk-events")
print("═" * 60)
print(f"\nTelemetry events sent: {events_sent}")
print(f"Dashboard URL: {console_url}")
print(f"\nEvery payment.initiated, guard_passed, guard_blocked,")
print(f"payment.completed, and payment.failed event from this")
print(f"notebook is now visible on the SDK Events page.")
print(f"\nYou will see:")
print(f"  • Green rows for completed payments")
print(f"  • Red rows for guard blocks (with reasons)")
print(f"  • Blue rows for initiated events")
print(f"  • Python language badges on every event")
print(f"  • Real-time auto-refresh every 5 seconds")

In [ ]:
# Cell 12 — Summary Statistics
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

entries = await client.ledger.query(wallet_id=wallet.id, limit=100)

status_counts = {}
for e in entries:
    s = e.status.value
    status_counts[s] = status_counts.get(s, 0) + 1

total_spent = sum(e.amount for e in entries if e.status.value == "completed")

print(f"Total payments attempted: {len(entries)}")
print(f"Total USDC spent:        {total_spent}")
print(f"Success rate:            {status_counts.get('completed', 0)}/{len(entries)} ({100 * status_counts.get('completed', 0) / max(len(entries), 1):.0f}%)")
print(f"Guard blocks:            {status_counts.get('blocked', 0)}")
print(f"Telemetry events sent:   {client.telemetry.events_sent}")

# Pie chart
colors = {
    "completed": "#22c55e",
    "blocked": "#ef4444",
    "failed": "#f97316",
    "pending": "#3b82f6",
}
labels = list(status_counts.keys())
sizes = list(status_counts.values())
chart_colors = [colors.get(l, "#6b7280") for l in labels]

fig, ax = plt.subplots(figsize=(6, 4), facecolor="#0a0a0a")
ax.set_facecolor("#0a0a0a")
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=chart_colors, autopct="%1.0f%%",
    textprops={"color": "white", "fontsize": 11},
)
for t in autotexts:
    t.set_fontweight("bold")
ax.set_title("Payment Outcomes", color="white", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n— Demo complete. Guards enforced policy, ledger captured everything, dashboard shows it all. —")